# Experiment 017: training-only checkpoint selection at 0.5B (Colab)

Evaluates one frozen pick per 006 seed (steps 400 / 450 / 250, chosen by the
trailing-50 training-reward rule BEFORE any checkpoint test evaluation. The question, stated narrowly: can this pre-specified
selector improve on the final checkpoints of the three observed runs?

Binding decision rule per seed: material = |overall delta| >= 2pt AND
holm-adjusted exact mcnemar p < .05; the bootstrap interval is effect-size
uncertainty only. Exploratory and terminal, the reportable 006 numbers stay
the final-checkpoint ones.

Approximately 2.5-3h, any a100, evaluation only. Resume-safe: finished evals skip, local
scratch is rebuilt from hashes on every pass.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 90.2   # from the resources panel
unit_rate = 6.77   # units per hour, the plan requires it per session
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '017_selector_test', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260726_130606 | cu_before = 90.2
Mounted at /content/drive
saved open session: 20260726_130606
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: any A100 (evaluation only)

In [2]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
assert 'A100' in gpu, 'pick an a100 runtime, evaluation stays on the study hardware family'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-4133855e-e8a1-dd1f-2fb4-3351b126860c)


## 2. Clone or update the repo (automatic)

In [ ]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

## 3. Install dependencies

In [ ]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


## 4. Check the Drive record

In [ ]:
import os
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')

## 5. Inventory and verify the three picked checkpoints

Everything the evaluation depends on gets checked before any gpu time burns:
the checkpoint exists at the exact step, the final provenance file and the
merged sft base exist, and the final predictions (the paired reference) are
present.

In [ ]:
PICKS = {0: (400, 0.385, 0.333), 1: (450, 0.340, 0.250), 2: (250, 0.403, 0.333)}
import hashlib, json, os
D = '/content/drive/MyDrive/inm363'

def sha_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

# recomputed picks and log hashes get carried into the eval cell's manifests
RECOMPUTED = {}
for s, (step, score, final_score) in PICKS.items():
    run = f'006_COLAB_FULL_B05_sft_grpo_seed{s}'
    ck = os.path.join(D, 'checkpoints', run, 'grpo', f'checkpoint-{step}')
    for fname in ('adapter_model.safetensors', 'adapter_config.json', 'trainer_state.json'):
        assert os.path.exists(os.path.join(ck, fname)), f'seed {s}: missing {fname} in {ck}'
    gs = json.load(open(os.path.join(ck, 'trainer_state.json')))['global_step']
    assert gs == step, f'seed {s}: trainer_state says step {gs}, expected {step}'

    # the provenance file must actually point at this seed's merged model, not
    # merely exist somewhere
    prov = json.load(open(os.path.join(D, 'checkpoints', run, 'grpo', 'adapter_base.json')))
    assert prov.get('kind') == 'local', f'seed {s}: provenance kind {prov.get("kind")!r}'
    assert prov.get('value') == f'checkpoints/{run}/sft_merged', f'seed {s}: provenance points at {prov.get("value")!r}'
    assert os.path.exists(os.path.join(D, 'checkpoints', run, 'sft_merged', 'config.json')), f'seed {s}: sft_merged missing'
    assert os.path.exists(os.path.join(D, 'results', run, 'grpo_preds.jsonl')), f'seed {s}: final preds missing'

    # the pick is recomputed from the training log, not trusted: trailing-50
    # mean of the accuracy reward, highest wins, ties to the later step
    log_path = os.path.join(D, 'results', run, 'grpo_trainlog.json')
    tl = json.load(open(log_path))
    per_step = {e['step']: e['rewards/accuracy_reward/mean'] for e in tl
                if 'rewards/accuracy_reward/mean' in e and 'step' in e}
    steps = sorted(per_step)
    scores = {}
    for cs in range(50, 501, 50):
        window = [per_step[x] for x in steps if cs - 49 <= x <= cs]
        assert len(window) == 50, f'seed {s}: window ending {cs} has {len(window)} logged steps, expected 50'
        scores[cs] = sum(window) / len(window)
    best = max(scores, key=lambda k: (scores[k], k))
    assert best == step, f'seed {s}: recomputed pick {best}, frozen pick {step}'
    RECOMPUTED[s] = {'exact_score': scores[best], 'exact_final_window_score': scores[500],
                     'trainlog_sha256': sha_file(log_path)}
    print(f'seed {s}: checkpoint-{step} verified (global_step {gs}), provenance content ok, pick recomputed ok')
print('inventory clean, all three evaluable, picks reproduced from the logs')

## 6. The three evaluations

Per seed: rebuild local scratch (adapter files only, the exact final
provenance file, hashes asserted to match the sources), evaluate through the
standard adapter path, write the selector manifest, delete scratch.

**If colab disconnects:** re-run the cell, finished evals skip in seconds.

In [7]:
import hashlib, json, os, shutil
RULE_TEXT = ('for each saved checkpoint step s in 50..500, score = mean training '
             'accuracy reward over the trailing 50 steps, highest wins, ties to later')

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
D = '/content/drive/MyDrive/inm363'
for s, (step, score, final_score) in PICKS.items():
    run = f'006_COLAB_FULL_B05_sft_grpo_seed{s}'
    tag = f'017_selector_B05_seed{s}_step{step}'
    ck = os.path.join(D, 'checkpoints', run, 'grpo', f'checkpoint-{step}')
    final_dir = os.path.join(D, 'checkpoints', run, 'grpo')
    out = os.path.join(D, 'results', tag)
    man_path = os.path.join(out, 'selector_manifest.json')
    print(f'===== {tag} =====')

    # local scratch, rebuilt from zero on every pass so leftovers are never trusted
    scratch = f'/content/017_scratch/seed{s}_step{step}'
    if os.path.isdir(scratch):
        shutil.rmtree(scratch)
    os.makedirs(scratch)
    hashes = {}
    for src_dir, fname, key in ((ck, 'adapter_model.safetensors', 'adapter'),
                                (ck, 'adapter_config.json', 'adapter_config'),
                                (final_dir, 'adapter_base.json', 'provenance')):
        src = os.path.join(src_dir, fname)
        dst = os.path.join(scratch, fname)
        shutil.copy2(src, dst)
        hashes[key + '_source'] = sha(src)
        hashes[key + '_copy'] = sha(dst)
        assert hashes[key + '_source'] == hashes[key + '_copy'], f'{fname}: copy hash differs'

    # The manifest is built BEFORE evaluation. Existing outputs are honoured only
    # if their manifest matches this one, otherwise the run stops: an old output
    # must never get relabelled as coming from this checkpoint
    expected = {'source_run': run, 'selected_step': step, 'selector_score': score,
                'final_window_score': final_score,
                'exact_score': RECOMPUTED[s]['exact_score'],
                'exact_final_window_score': RECOMPUTED[s]['exact_final_window_score'],
                'trainlog_sha256': RECOMPUTED[s]['trainlog_sha256'],
                'source_checkpoint': ck,
                'trainer_state_global_step': json.load(open(os.path.join(ck, 'trainer_state.json')))['global_step'],
                'hashes': hashes,
                'reference_preds_sha256': sha(os.path.join(D, 'results', run, 'grpo_preds.jsonl')),
                'selector_rule': RULE_TEXT}
    if os.path.exists(os.path.join(out, 'grpo_eval.json')):
        assert os.path.exists(man_path), f'{tag}: outputs exist without a selector manifest, delete {out} and rerun'
        existing = json.load(open(man_path))
        for k in ('source_run', 'selected_step', 'hashes', 'selector_rule', 'trainlog_sha256',
                  'reference_preds_sha256', 'trainer_state_global_step', 'exact_score', 'exact_final_window_score'):
            assert existing.get(k) == expected[k], f'{tag}: manifest field {k} does not match, refusing to relabel'
        print('existing outputs match the expected manifest, evaluator will skip')
    else:
        os.makedirs(out, exist_ok=True)
        with open(man_path, 'w') as f:
            json.dump(expected, f, indent=2)

    !python scripts/eval_adapter.py --config configs/006_colab_full_b05_seed{s}.yaml --adapter {scratch} --condition grpo --out {out} --seed {s}
    if _exit_code != 0:
        raise SystemExit(f'{tag} failed with exit code {_exit_code}')

    # completeness gates before this artifact counts
    ev = json.load(open(os.path.join(out, 'grpo_eval.json')))
    lp = json.load(open(os.path.join(out, 'grpo_logprob.json')))
    assert ev['n_items'] == 1884 and lp['n_items'] == 1884, f'{tag}: item counts wrong'
    em = json.load(open(os.path.join(out, 'grpo_eval_manifest.json')))
    assert em.get('completed') is True, f'{tag}: eval manifest not completed'
    shutil.rmtree(scratch)
    print(f'{tag} done and verified, scratch removed')

===== 017_selector_B05_seed0_step400 =====
README.md: 100% 53.2k/53.2k [00:00<00:00, 92.0MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 199MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B/s]
high_school_psychology/test-00000-of-000(…): downloading bytes: 100% 91.9k/91.9k [00:00<00:00, 141kB/s, 9.10kB/s  ]
high_school_psychology/test-00000-of-000(…): reconstructing file: 100% 92.8k/92.8k [00:00<00:00, 142kB/s, 9.18kB/s  ]

high_school_psychology/validation-00000-(…): downloading bytes:   0% 0.00/15.2k [00:00<?, ?B/s]
high_school_psychology/validation-00000-(…): downloading bytes: 100% 13.4k/13.4k [00:00<00:00, 22.3kB/s, 1.32kB/s  ]
high_school_psychology/validation-00000-(…): reconstructing file: 100% 15.2k/15.2k [00:00<00:00, 25.4kB/s, 1.51kB/s  ]

high_school_psychology/dev-00000-of-0000(…): downloading bytes:   0% 0.00/5.18k [00:00<?, ?B/s]
high_school_psychology/dev-00000-of-0000(…): downloading bytes: 100% 3.21k/3.21k [00:0

## 7. The pre-registered comparison

Overall picked-minus-final per seed, holm-adjusted exact mcnemar as the
binding gate, bootstrap interval as effect-size uncertainty, tiers as
diagnostics. Cross-era alignment throughout, tiers rebuilt from the current
subject mapping.

In [8]:
PICKS = {0: (400, 0.385, 0.333), 1: (450, 0.340, 0.250), 2: (250, 0.403, 0.333)}
import json, os, sys
sys.path.insert(0, os.getcwd())
from analysis.stats import _read_preds, align_pred_rows, mcnemar_exact, paired_delta_ci, holm
from data.subjects import tier_of, TIERS

D = '/content/drive/MyDrive/inm363'

def probe_acc(path):
    rs = _read_preds(path)
    per = {t: [r['correct'] for r in rs if tier_of(r['subject']) == t] for t in TIERS}
    return sum(r['correct'] for r in rs) / len(rs), {t: sum(v) / len(v) for t, v in per.items()}

rows_out = []
for s, (step, _, _) in PICKS.items():
    run = f'006_COLAB_FULL_B05_sft_grpo_seed{s}'
    tag = f'017_selector_B05_seed{s}_step{step}'
    pdir, fdir = os.path.join(D, 'results', tag), os.path.join(D, 'results', run)
    picked = _read_preds(os.path.join(pdir, 'grpo_preds.jsonl'))
    final = _read_preds(os.path.join(fdir, 'grpo_preds.jsonl'))
    ra, rb = align_pred_rows(picked, final)   # old and new subject orders realigned
    ca = [r['correct'] for r in ra]
    cb = [r['correct'] for r in rb]
    delta = sum(ca) / len(ca) - sum(cb) / len(cb)
    n01 = sum(1 for x, y in zip(ca, cb) if not x and y)
    n10 = sum(1 for x, y in zip(ca, cb) if x and not y)
    d, lo, hi = paired_delta_ci(cb, ca)   # picked minus final
    tiers = {}
    for t in TIERS:
        xs = [(x, y) for x, y, r in zip(ca, cb, ra) if tier_of(r['subject']) == t]
        tiers[t] = (sum(x for x, y in xs) / len(xs), sum(y for x, y in xs) / len(xs))
    pev = json.load(open(os.path.join(pdir, 'grpo_eval.json')))
    fev = json.load(open(os.path.join(fdir, 'grpo_eval.json')))
    ppr = probe_acc(os.path.join(pdir, 'grpo_logprob_preds.jsonl'))
    fpr = probe_acc(os.path.join(fdir, 'grpo_logprob_preds.jsonl'))
    rows_out.append({'seed': s, 'step': step, 'picked_acc': sum(ca) / len(ca),
                     'final_acc': sum(cb) / len(cb), 'delta': delta, 'p_raw': mcnemar_exact(n01, n10),
                     'ci': (lo, hi), 'tiers': tiers,
                     'diag': {'fmt': (pev['format_compliance'], fev['format_compliance']),
                              'len': (pev['mean_len'], fev['mean_len']),
                              'xfail': (pev['extraction_failure_rate'], fev['extraction_failure_rate']),
                              'probe_overall': (ppr[0], fpr[0]), 'probe_tiers': (ppr[1], fpr[1])}})

adj = holm([r['p_raw'] for r in rows_out])
print(f"tier mapping used: {TIERS}")
cats = {}
for r, pa in zip(rows_out, adj):
    material = abs(r['delta']) >= 0.02 and pa < 0.05
    cat = ('improvement' if r['delta'] > 0 else 'decline') if material else 'none'
    cats[r['seed']] = cat
    label = {'improvement': 'material improvement', 'decline': 'material decline', 'none': 'no clear material difference'}[cat]
    dg = r['diag']
    print(f"seed {r['seed']} (step {r['step']}): picked {r['picked_acc']:.3f} vs final {r['final_acc']:.3f}  "
          f"delta {r['delta']:+.3f}  holm p {pa:.3g}  ci [{r['ci'][0]:+.3f}, {r['ci'][1]:+.3f}]  -> {label}")
    for t, (a, b) in r['tiers'].items():
        print(f"   {t}: picked {a:.3f} vs final {b:.3f}")
    print(f"   fmt {dg['fmt'][0]:.3f} vs {dg['fmt'][1]:.3f}  len {dg['len'][0]:.0f} vs {dg['len'][1]:.0f}  "
          f"xfail {dg['xfail'][0]:.3f} vs {dg['xfail'][1]:.3f}  probe {dg['probe_overall'][0]:.3f} vs {dg['probe_overall'][1]:.3f}")

# the aggregate verdict, straight from the interpretation table in the plan
n_imp = sum(1 for c in cats.values() if c == 'improvement')
n_dec = sum(1 for c in cats.values() if c == 'decline')
# mixed signs take precedence over the count rules, per the plan
if n_imp >= 1 and n_dec >= 1:
    verdict = 'inconsistent signs, reported seed by seed'
elif n_imp >= 2:
    verdict = 'material improvement in at least two seeds: useful across most observed artifacts'
elif n_dec >= 2:
    verdict = 'material decline in at least two seeds: this selector was generally harmful across these artifacts'
elif n_imp == 1 and cats.get(1) == 'improvement':
    verdict = 'material improvement only in seed 1: rescues the weak artifact only'
elif n_imp == 1:
    verdict = 'material improvement in exactly one other seed: isolated benefit, no pattern'
elif n_dec == 1:
    verdict = 'material decline in exactly one seed: isolated harm, reported seed by seed'
elif n_imp == 0 and n_dec == 0:
    verdict = 'no material result in any seed: no evidence of benefit from this heuristic'
else:
    verdict = 'inconclusive, reported seed by seed'
print()
print('VERDICT (interpretation table):', verdict)

tier mapping used: {'easy': ['high_school_psychology', 'marketing', 'us_foreign_policy'], 'medium': ['high_school_geography', 'philosophy', 'business_ethics'], 'hard': ['high_school_mathematics', 'formal_logic']}
seed 0 (step 400): picked 0.452 vs final 0.462  delta -0.010  holm p 0.18  ci [-0.022, +0.001]  -> no clear material difference
   easy: picked 0.556 vs final 0.575
   medium: picked 0.432 vs final 0.442
   hard: picked 0.253 vs final 0.245
   fmt 0.939 vs 0.941  len 66 vs 66  xfail 0.040 vs 0.037  probe 0.564 vs 0.568
seed 1 (step 450): picked 0.307 vs final 0.315  delta -0.008  holm p 0.18  ci [-0.020, +0.003]  -> no clear material difference
   easy: picked 0.371 vs final 0.379
   medium: picked 0.271 vs final 0.284
   hard: picked 0.220 vs final 0.222
   fmt 0.888 vs 0.898  len 111 vs 107  xfail 0.069 vs 0.059  probe 0.564 vs 0.563
seed 2 (step 250): picked 0.365 vs final 0.506  delta -0.141  holm p 1.69e-33  ci [-0.163, -0.119]  -> material decline
   easy: picked 0.461 v

## 8. Close the units record

In [2]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 71.33   # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

drive not mounted, run the mount cell first
